# C14_E01 — Pseudo-PLC: bloque PID con ciclo de scan

**Capítulo:** 14 — Instrumentación, arquitecturas industriales, SCADA/DCS, IoT e IIoT  
**Identificador:** `C14_E01`  
**Objetivo pedagógico:** Mostrar el patrón industrial de PID en PLC con bumpless transfer.  
**Librerías:** estándar Python (numpy opcional)

## Ejemplo industrial

Bloque PID en TIA Portal / RSLogix / DeltaV: estado interno y modos AUTO/MANUAL.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Bloque PID en estilo PLC con ciclo de scan

In [1]:
class PIDPLC:
    """PID incremental en estilo PLC, con ciclo de scan determinístico."""
    def __init__(self, Kp, Ti, Td, Ts, umin=0.0, umax=100.0):
        self.Kp, self.Ti, self.Td, self.Ts = Kp, Ti, Td, Ts
        self.umin, self.umax = umin, umax
        self.e1 = 0.0; self.e2 = 0.0; self.u = 0.0; self.modo = "AUTO"
    def step(self, sp, pv):
        e = sp - pv
        if self.modo == "MANUAL":
            return self.u  # mantiene salida
        du = self.Kp*((e - self.e1)
                      + (self.Ts/self.Ti)*e
                      + (self.Td/self.Ts)*(e - 2*self.e1 + self.e2))
        u_new = self.u + du
        u_sat = max(self.umin, min(self.umax, u_new))
        self.u = u_sat; self.e2 = self.e1; self.e1 = e
        return u_sat
    def to_manual(self, u_manual):
        """Bumpless transfer hacia manual."""
        self.modo = "MANUAL"; self.u = u_manual
    def to_auto(self):
        """Bumpless transfer hacia automático: continúa desde la última u."""
        self.modo = "AUTO"; self.e1 = 0.0; self.e2 = 0.0

## 2. Simulación de un ciclo (sin figura)

In [2]:
pid = PIDPLC(Kp=0.6, Ti=10.0, Td=0.0, Ts=0.2, umin=0, umax=100)
sp = 50.0; pv = 30.0
for _ in range(5):
    u = pid.step(sp, pv)
    print(f"PV={pv:.2f}  →  u={u:.2f}")
    pv += 0.1*(u - pv)*pid.Ts
print()
pid.to_manual(40.0); print("Modo MANUAL  →  u =", pid.step(sp, pv))
pid.to_auto();      print("Modo AUTO    →  u =", pid.step(sp, pv))

PV=30.00  →  u=12.24
PV=29.64  →  u=12.70
PV=29.31  →  u=13.15
PV=28.98  →  u=13.60
PV=28.67  →  u=14.04

Modo MANUAL  →  u = 40.0
Modo AUTO    →  u = 53.23010558233909


## 3. Interpretación

El bloque mantiene `u`, `e1`, `e2` como estado interno y soporta cambio AUTO ↔ MANUAL bumpless: en MANUAL devuelve la `u` impuesta por operador; al volver a AUTO continúa desde esa `u` sin saltos. Este patrón es el utilizado en bloques PID industriales reales (Honeywell PIDA, Siemens FB41, Emerson PID, ABB AC800M).